# 03 · Repeat tickets and repeat-contact pairings

This notebook replicates the Repeat tickets and Repeat-contact pairings paragraphs of the report's Results section.

> Repeat tickets: Of 6,665 eligible tickets, 23.6% repeat within 30 days and 9.5% within 7 days, 1.53 and 2.30 times random timing's shares. Without monitoring tickets, the 30-day share is 22.0%.
>
> Repeat-contact pairings: Of the 1,574 repeats, 31% are the same Category, 10% the same request group with another Category, and 32% a different group. The other 28% involve monitoring, No Trouble Found, or a blank Category. Over all pairs of a machine's tickets, 46% of pairs a day or less apart share a Category vs. 27% of pairs over six months apart. No model or telemetry type with 100 or more eligible tickets clearly stands out from the fleet.

We also check two numbers that the report's Definitions give for its last statement, under Clearly stands out: the 12 models and 11 telemetry types tested.

**How it works.** `followups.py` pairs each eligible ticket with the next unplanned ticket on its machine, classifies the repeats, gives each ticket its chance of a repeat under random timing, and compares each model and telemetry type with the fleet using a bootstrap over accounts. We recompute every number from its per-ticket table, `followups/rp_pairs.csv`, with three exceptions: the shares over all pairs of a machine's tickets come from the counts by gap in `followups/rp_same_by_gap.csv`, the bootstrap intervals and p-values come from `followups/rp_by_model.csv` and `followups/rp_by_type.csv`, and the threshold of 100 eligible tickets is the script's setting, which we read from `followups/followups_summary.json`. As checks, we compare the counts of eligible tickets and repeats with those that two other scripts make from the case export, we rebuild the random-timing chances from each machine's tickets and days in the period, and we assert that our recomputed columns match the script's.

**Re-run.** With `RERUN = True`, we re-run `verification/followups.py`, which takes about 20 s. It rewrites its 22 `ag_*` and `rp_*` tables and `followups_summary.json` in the `followups/` folder of a scratch copy of `verification/outputs/`, and we then compare each of the 23 files with the committed one. `docs/09_followups_protocol.md` (§5 and §12) and `docs/10_followups_results.md` (items 8 to 10 and §3) describe this part of the analysis.

**Used as committed, without re-running.** The script also reads `fu/fu_ticket_last10.csv` and `fu/fu_devices.csv` to give each machine its telemetry type. They are the ticket-to-device map that `db_followup.py` wrote from the telemetry database, which the notebooks do not re-read. Before any result, the script compares its run with committed outputs of `fleet_exposure.py`, `reproduce_sql.py`, `reproduce_cleanroom.py`, and `audit_case_logic.py`, such as the fleet totals, the per-machine table, the repeat counts, and the gap bins, and it stops if any of them differs. To check the counts, this notebook reads two of those outputs as well: `repro_sql_repeat_summary.csv` from `reproduce_sql.py` (notebook 08 re-runs it) and `audit_case_repeat_gap_bins.csv` from `audit_case_logic.py`. To check the random-timing chances, it reads `fleet/fx_machine_flags.csv` and `fleet/fx_summary.json` from `fleet_exposure.py` (notebook 04 re-runs it) and `htm_machine_counts.csv` from `high_ticket_machines.py` (notebook 02 re-runs it).

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'followups.py')
R.compare(OUT)

followups.py: exit status 0, 22 s
    67/67 checks pass; 21 s
23 files written: 23 identical.


,file written by the re-run,compared with the committed file
0,followups/ag_band_rates.csv,identical
1,followups/ag_by_model.csv,identical
2,followups/ag_by_status.csv,identical
3,followups/ag_by_type.csv,identical
4,followups/ag_decomposition.csv,identical
5,followups/ag_halves.csv,identical
6,followups/ag_install_pooled.csv,identical
7,followups/ag_install_year.csv,identical
8,followups/ag_support.csv,identical
9,followups/ag_tests.csv,identical


## 6,665 eligible tickets: 23.6% repeat within 30 days and 9.5% within 7 days

An eligible ticket is an unplanned ticket opened before Aug 2nd, 2026, and it repeats when the machine's next unplanned ticket opens within 30 days, or within 7 days for the 7-day share, which has the same denominator. `followups/rp_pairs.csv` has one row per eligible ticket with the gap in days to its machine's next ticket, so we count the rows and flag the gaps of 0 to 30 and 0 to 7 days. We assert that our flags match the script's `repeat_30` and `repeat_7` columns, whose longest flagged gap and shortest gap left out show the windows the script used. The table could leave out eligible tickets without any of this changing, so we also assert that our counts equal those of two scripts that count from the case export in runs of their own: `reproduce_sql.py`, which runs the midterm analysis's SQL (`repro_sql_repeat_summary.csv`), and `audit_case_logic.py`, which counts the repeats by gap (`audit_case_repeat_gap_bins.csv`).

In [3]:
import math
import numpy as np

pairs = R.read_csv(OUT, 'followups/rp_pairs.csv')  # one row per eligible ticket, with its machine's next ticket
WINDOW, SHORT = 30, 7  # days, as the report's Definitions state
CUTOFF = '2026-08-02'  # eligible tickets open before Aug 2nd, 2026, as the report's Definitions state

n_eligible, n_machines = len(pairs), pairs.machine_id.nunique()
within_30 = pairs.gap_days.between(0, WINDOW)  # a ticket with no next ticket has no gap and does not repeat
within_7 = pairs.gap_days.between(0, SHORT)
share_30, share_7 = within_30.mean(), within_7.mean()
assert (pairs.opened_date < CUTOFF).all(), 'an eligible ticket opens on or after the cutoff'
assert (within_30 == pairs.repeat_30).all() and (within_7 == pairs.repeat_7).all(), "our flags differ from the script's"

# Cross-check: the same counts from the case export, by reproduce_sql.py (the midterm SQL) and audit_case_logic.py
sql = R.read_csv(OUT, 'repro_sql_repeat_summary.csv').iloc[0]
sql_counts = (sql.eligible_tickets, sql.machines, sql.repeat_tickets)
assert sql_counts == (n_eligible, n_machines, within_30.sum()), 'reproduce_sql.py counts differently'
cumulative = R.read_csv(OUT, 'audit_case_repeat_gap_bins.csv').set_index('gap_bin').cum_repeats  # repeats up to a gap
up_to = {d: cumulative[cumulative.index.str.endswith(f',{d}d]')].item() for d in (SHORT, WINDOW)}  # '(2d,7d]', ...
assert (up_to[SHORT], up_to[WINDOW]) == (within_7.sum(), within_30.sum()), 'audit_case_logic.py counts differently'
print(f'{n_eligible:,} eligible tickets on {n_machines:,} machines, all opened before Aug 2nd, 2026, with '
      f'{int(within_30.sum()):,} repeats within 30 days and {int(within_7.sum())} within 7 days. Our flags match the '
      "script's, and reproduce_sql.py and audit_case_logic.py count the same from the case export.")


def script_window(flag):
    """The window behind one of the script's flags, in whole days: its longest flagged gap rounded up, provided that
    every next ticket the flag leaves out is further away."""
    days = math.ceil(pairs.gap_days[flag].max())
    left_out = pairs.gap_days[pairs.next_case_id.notna() & ~flag]
    return days if left_out.min() > days else math.nan


window_30, window_7 = script_window(pairs.repeat_30), script_window(pairs.repeat_7)
has_next = pairs.next_case_id.notna()
pd.DataFrame({'repeats': [int(within_30.sum()), int(within_7.sum())],
              'share of eligible tickets': [R.pct(share_30, 1), R.pct(share_7, 1)],
              'longest gap the script flags (days)': [pairs.gap_days[f].max() for f in (pairs.repeat_30, pairs.repeat_7)],
              'shortest gap it leaves out (days)': [pairs.gap_days[has_next & ~f].min()
                                                    for f in (pairs.repeat_30, pairs.repeat_7)],
              "script's window (days)": [window_30, window_7]},
             index=['within 30 days', 'within 7 days']).round(3)

6,665 eligible tickets on 3,495 machines, all opened before Aug 2nd, 2026, with 1,574 repeats within 30 days and 631 within 7 days. Our flags match the script's, and reproduce_sql.py and audit_case_logic.py count the same from the case export.


,repeats,share of eligible tickets,longest gap the script flags (days),shortest gap it leaves out (days),script's window (days)
within 30 days,1574,23.6%,29.997,30.014,30
within 7 days,631,9.5%,7.000,7.002,7


## 1.53 and 2.30 times random timing's shares

Under random timing, a machine's n unplanned tickets fall uniformly at random over its L days in the period (from its install or first ticket, whichever is earlier, but no earlier than Aug 1st, 2025, and over the whole period for serials not on IPA's list), so an eligible ticket's next ticket falls within d days with chance 1 − (1 − d/L)^(n − 1). `rp_pairs.csv` stores this chance for each ticket (`null_p30` and `null_p7`), random timing's share is their sum divided by the eligible tickets, and the ratio is the observed share divided by it.

We rebuild the chances from each fleet machine's tickets and months in `fleet/fx_machine_flags.csv`. The fleet leaves out the serials not on IPA's list and the listed machines on the third-party Vogeltronix plan. Random timing spreads the tickets of a serial not on IPA's list over the whole period, as it does for a listed machine installed by Aug 1st, 2025. So we rebuild the other machines' chances from the whole period and each machine's unplanned tickets in `htm_machine_counts.csv`, where `high_ticket_machines.py` counts them for every machine with a ticket. We first assert that the two tables give each fleet machine the same tickets. To split the other machines' eligible tickets between the two groups, we take the eligible tickets on all listed machines from the script's row for listed machines only in `followups/rp_null.csv` and subtract those in the fleet's table.

The Monte Carlo check needs the script's 1,000 simulations, so we read its result from `followups/followups_summary.json`. It compares the simulations' 30-day share with the exact chances over the same simulated tickets. We also show the median and the middle 95% of the simulations' shares from `rp_null.csv`. Each simulation's share is over its own eligible tickets, the ones its random times put before Aug 2nd, so the median differs slightly from the exact share, which is over the observed eligible tickets.

In [4]:
expected_30, expected_7 = pairs.null_p30.sum(), pairs.null_p7.sum()  # expected repeats under random timing
chance_30, chance_7 = expected_30 / n_eligible, expected_7 / n_eligible
ratio_30, ratio_7 = share_30 / chance_30, share_7 / chance_7


def closed_form(n, L, d):
    """The chance that an eligible ticket's next ticket falls within d days, as followups.py computes it. It is 0 for a
    machine with one ticket and for one with 30 days or less in the period, which can have no eligible ticket."""
    return np.where((n > 1) & (L > WINDOW), 1 - (1 - d / L) ** (n - 1), 0.0)


# Check 1: the fleet's machines, from each one's tickets n and days L in the period (fleet_exposure.py's table)
flags = R.read_csv(OUT, 'fleet/fx_machine_flags.csv')  # one row per fleet machine with a ticket
# fx_machine_flags.csv's serial_id and rp_pairs.csv's machine_id are the same S##### pseudonyms
chances = pairs.groupby('machine_id')[['null_p30', 'null_p7']].first()
machines = chances.join(flags.set_index('serial_id')[['tickets', 'months']], how='inner')
n, L = machines.tickets, machines.months * 365.25 / 12  # fleet_exposure.py's months are days / (365.25 / 12)
largest_difference = max(float(np.abs(closed_form(n, L, d) - machines[col]).max())
                         for d, col in ((WINDOW, 'null_p30'), (SHORT, 'null_p7')))
assert largest_difference < 1e-9, 'the closed form differs from the stored chances'

# Check 2: the other machines, from each one's tickets n (high_ticket_machines.py's table) and the whole period as L
tickets = R.read_csv(OUT, 'htm_machine_counts.csv').set_index('machine_id').unplanned_tickets  # each machine with one
assert (tickets.loc[flags.serial_id].to_numpy() == flags.tickets.to_numpy()).all(), 'the two tables count differently'
summary = R.read_json(OUT, 'followups/followups_summary.json')
first_day, last_day = pd.to_datetime(summary['definitions']['period'])
whole_period = (last_day - first_day).days + 1  # days
outside = chances.drop(machines.index)
n_outside = tickets.loc[outside.index]
largest_outside = max(float(np.abs(closed_form(n_outside, whole_period, d) - outside[col]).max())
                      for d, col in ((WINDOW, 'null_p30'), (SHORT, 'null_p7')))
assert largest_outside < 1e-9, "the whole period does not give the other machines' stored chances"

# Which of the other machines' eligible tickets are on listed machines: the script counts the listed machines' tickets
null_rows = R.read_csv(OUT, 'followups/rp_null.csv').set_index(['variant', 'window'])
in_table = pairs.machine_id.isin(machines.index)
listed_eligible = null_rows.loc[('own window, uniform, listed machines only', '30 days'), 'eligible']
on_plan = int(listed_eligible) - int(in_table.sum())
fleet_summary = R.read_json(OUT, 'fleet/fx_summary.json')
# fleet_exposure.py: every ticket on a listed machine is on a fleet machine (in the table) or on the third-party plan
third_party = fleet_summary['third_party']['tickets']
on_listed = fleet_summary['match']['unplanned tickets on listed machines']
assert on_listed == flags.tickets.sum() + third_party, 'a listed ticket is in neither the table nor the plan'
assert 0 <= on_plan <= third_party, 'more eligible tickets on listed machines outside the table than on the plan'
print(f'The closed form gives the stored chances of the {len(machines):,} fleet machines in fx_machine_flags.csv '
      f'({in_table.sum():,} eligible tickets) to within {largest_difference:.0e}. From their tickets in '
      f'htm_machine_counts.csv and the whole period of {whole_period} days, it gives those of the other {len(outside)} '
      f'machines ({(~in_table).sum()} eligible tickets) to within {largest_outside:.0e}. Of these eligible tickets, '
      f"{(~in_table).sum() - on_plan} are on serials not on IPA's list and {on_plan} on listed machines on the "
      'third-party plan.')

# The Monte Carlo check, which needs the simulations: the script's own comparison over the same simulated tickets
b4 = next(c for c in summary['checks']['list'] if c['check'].startswith('B4: the simulation agrees'))
assert b4['status'] == 'PASS', b4
sim = null_rows.loc['own window, uniform'].reindex(['30 days', '7 days'])
n_sim = summary['n_sim']
print(f"followups.py's Monte Carlo check: over the same simulated tickets, the {n_sim:,} simulations' 30-day share is "
      f"within {100 * b4['actual']:.2f}% of the exact share ({b4['status']}). Each simulation's share is over its own "
      f'eligible tickets ({sim.simulated_eligible_mean.iloc[0]:,.0f} on average vs. {n_eligible:,} observed), so the '
      'median below differs slightly from the exact share.')
pd.DataFrame({'observed': [R.pct(share_30, 1), R.pct(share_7, 1)],
              'random timing, exact': [R.pct(chance_30, 2), R.pct(chance_7, 2)],
              'observed ÷ random timing': [R.num(ratio_30, 2), R.num(ratio_7, 2)],
              f'median of {n_sim:,} simulations': [f'{x:.2f}%' for x in sim.null_pct_sim_median],
              'middle 95% of simulations': [f'{lo:.2f}–{hi:.2f}%'
                                            for lo, hi in zip(sim.null_pct_sim_p2_5, sim.null_pct_sim_p97_5)]},
             index=['within 30 days', 'within 7 days'])

The closed form gives the stored chances of the 3,269 fleet machines in fx_machine_flags.csv (6,234 eligible tickets) to within 5e-11. From their tickets in htm_machine_counts.csv and the whole period of 396 days, it gives those of the other 226 machines (431 eligible tickets) to within 4e-11. Of these eligible tickets, 428 are on serials not on IPA's list and 3 on listed machines on the third-party plan.
followups.py's Monte Carlo check: over the same simulated tickets, the 1,000 simulations' 30-day share is within 0.05% of the exact share (PASS). Each simulation's share is over its own eligible tickets (6,595 on average vs. 6,665 observed), so the median below differs slightly from the exact share.


,observed,"random timing, exact",observed ÷ random timing,"median of 1,000 simulations",middle 95% of simulations
within 30 days,23.6%,15.41%,1.53,15.28%,14.60–16.04%
within 7 days,9.5%,4.11%,2.30,4.07%,3.62–4.52%


## Without monitoring tickets, the 30-day share is 22.0%

The report removes monitoring tickets from each machine's sequence, so they leave the eligible tickets, and a ticket followed by a monitoring ticket is paired with the next ticket that is not one. In `rp_pairs.csv`, every monitoring ticket that follows an eligible ticket is eligible too and has its own row (the code checks this), so we start from each ticket that is not a monitoring ticket, step past the monitoring tickets that follow it, add up the gaps, and count the next tickets within 30 days. `followups.py` makes the same count from the case export as one of its checks, and we assert that our count equals the one it records in `followups/followups_summary.json`.

In [5]:
by_case = pairs.set_index('case_id')
chain = pairs.loc[~pairs.index_monitoring, ['next_case_id', 'gap_days', 'next_monitoring']].copy()
while chain.next_monitoring.any():  # step past each monitoring follow-up to the ticket after it
    hop = chain.next_monitoring
    if not chain.next_case_id[hop].isin(by_case.index).all():
        raise ValueError('A monitoring follow-up has no row of its own in rp_pairs.csv.')
    step = by_case.loc[chain.next_case_id[hop].to_numpy()]
    chain.loc[hop, 'gap_days'] = chain.gap_days[hop].to_numpy() + step.gap_days.to_numpy()
    chain.loc[hop, 'next_case_id'] = step.next_case_id.to_numpy()
    chain.loc[hop, 'next_monitoring'] = step.next_monitoring.to_numpy()

eligible_no_monitoring = len(chain)
repeat_no_monitoring = chain.gap_days.between(0, WINDOW)
share_no_monitoring = repeat_no_monitoring.mean()
monitoring_index = pairs.index_monitoring
lost = int((within_30[chain.index] & ~repeat_no_monitoring).sum())  # skipping a ticket can only lengthen a gap

pin = next(c for c in summary['checks']['list'] if c['check'].startswith('pin slide 5'))  # its count on the case export
assert pin['status'] == 'PASS' and pin['actual'] == [int(repeat_no_monitoring.sum()), eligible_no_monitoring], pin
print(f'{int(repeat_no_monitoring.sum()):,} repeats of {eligible_no_monitoring:,} eligible tickets without monitoring '
      f"tickets ({R.pct(share_no_monitoring, 1)}), as followups.py counts them from the case export.")
pd.DataFrame({'eligible tickets': [n_eligible, -int(monitoring_index.sum()), 0, eligible_no_monitoring],
              'repeats within 30 days': [int(within_30.sum()), -int(within_30[monitoring_index].sum()), -lost,
                                         int(repeat_no_monitoring.sum())]},
             index=['all eligible tickets', 'less monitoring tickets as the index',
                    'less repeats whose only follow-ups within 30 days are monitoring tickets',
                    'without monitoring tickets'])

1,438 repeats of 6,529 eligible tickets without monitoring tickets (22.0%), as followups.py counts them from the case export.


,eligible tickets,repeats within 30 days
all eligible tickets,6665,1574
less monitoring tickets as the index,-136,-48
less repeats whose only follow-ups within 30 days are monitoring tickets,0,-88
without monitoring tickets,6529,1438


## The 1,574 repeats by pairing class

Each repeat takes the first class that applies: a monitoring ticket involved (either ticket), No Trouble Found involved, a blank Category (`Not categorised` in the table), the same Category, the same request group with another Category, or a different group. We classify the repeats in `rp_pairs.csv` from the two tickets' monitoring flags, Categories, and request groups, assert that our classes match the script's `pair_class` column, and divide each class by the repeats. The report's 28% is the first three classes together.

In [6]:
CLASSES = ['Monitoring involved', 'No Trouble Found involved', 'Category blank', 'Same Category',
           'Same group, other Category', 'Different group']
rep = pairs[within_30]
n_repeats = len(rep)
pair_class = pd.Series(np.select(
    [rep.index_monitoring | rep.next_monitoring,
     rep.index_category.eq('No Trouble Found') | rep.next_category.eq('No Trouble Found'),
     rep.index_category.eq('Not categorised') | rep.next_category.eq('Not categorised'),
     rep.index_category.eq(rep.next_category),
     rep.index_group.eq(rep.next_group)], CLASSES[:5], CLASSES[5]), index=rep.index)
assert (pair_class == rep.pair_class).all(), "our classes differ from the script's"
print(f"Our classes match the script's pair_class for all {n_repeats:,} repeats.")

class_counts = pair_class.value_counts().reindex(CLASSES, fill_value=0)
class_share = class_counts / n_repeats
share_process = class_share[CLASSES[:3]].sum()  # monitoring, No Trouble Found, or a blank Category
table = pd.DataFrame({'repeats': class_counts, 'share': class_share.map(lambda s: R.pct(s, 1))})
table.loc['Monitoring, No Trouble Found, or a blank Category'] = [int(class_counts[CLASSES[:3]].sum()),
                                                                   R.pct(share_process, 1)]
table.loc['All repeats'] = [n_repeats, R.pct(class_share.sum(), 1)]
table

Our classes match the script's pair_class for all 1,574 repeats.


,repeats,share
Monitoring involved,173,11.0%
No Trouble Found involved,201,12.8%
Category blank,61,3.9%
Same Category,490,31.1%
"Same group, other Category",150,9.5%
Different group,499,31.7%
"Monitoring, No Trouble Found, or a blank Category",435,27.6%
All repeats,1574,100.0%


## All pairs of a machine's tickets: 46% a day or less apart vs. 27% over six months apart

These are all pairs of unplanned tickets on one machine over the whole period, consecutive or not, in which both tickets have a recorded Category (No Trouble Found counts as one) and neither is a monitoring ticket. `followups.py` builds them from the case export and writes only their counts by gap, so `followups/rp_same_by_gap.csv` is the most granular committed table here (`rp_pairs.csv` has each eligible ticket's next ticket only), and the share is the pairs that share a Category divided by the pairs in the bin. The report words the shortest and longest bins, `1 day or less` and `over 180 days`, as "a day or less" and "over six months" (six 30-day months).

In [7]:
import re

by_gap = R.read_csv(OUT, 'followups/rp_same_by_gap.csv')  # one row per gap bin: pairs, and pairs that share a Category
by_gap['share'] = by_gap.same_category_pairs / by_gap.pairs
nearest, farthest = by_gap.iloc[0], by_gap.iloc[-1]
share_nearest, share_farthest = nearest.share, farthest.share
nearest_days = int(re.search(r'\d+', nearest.gap_bin).group())    # '1 day or less' -> 1
farthest_days = int(re.search(r'\d+', farthest.gap_bin).group())  # 'over 180 days' -> 180


def in_words(days):
    """A gap as the report words it: 1 day is 'a day', and a whole number of 30-day months is written in words."""
    if days == 1:
        return 'a day'
    words = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten', 'eleven', 'twelve']
    return f'{words[days // 30]} months' if days % 30 == 0 and days // 30 < len(words) else f'{days} days'


print(f'{int(by_gap.pairs.sum()):,} pairs. The share falls from {R.pct(share_nearest)} at {in_words(nearest_days)} or '
      f'less to {R.pct(share_farthest)} at over {in_words(farthest_days)}.')
by_gap.set_index('gap_bin')[['pairs', 'same_category_pairs', 'share']].assign(share=by_gap.share.map(R.pct).to_numpy())

6,980 pairs. The share falls from 46% at a day or less to 27% at over six months.


,pairs,same_category_pairs,share
gap_bin,,,
1 day or less,63,29,46%
over 1 to 7 days,522,228,44%
over 7 to 30 days,1291,466,36%
over 30 to 90 days,1967,678,34%
over 90 to 180 days,1730,528,31%
over 180 days,1407,380,27%


## No model or telemetry type with 100 or more eligible tickets clearly stands out

For each model and telemetry type with at least 100 eligible tickets, the script divides observed repeats by random timing's expected repeats (the sum of `null_p30`) and then by the fleet's 1.53, and the unit clearly stands out when this ratio's 95% interval over accounts excludes 1 and its Benjamini–Hochberg q is under 0.05 (the No device match and Unknown rows are left untested, since they are not telemetry types). We count each unit's eligible tickets, repeats, and expected repeats in `rp_pairs.csv`, take the threshold of 100 from `followups_summary.json`, and recompute the ratio. The intervals and p-values need the script's bootstrap, so we read them from `followups/rp_by_model.csv` and `followups/rp_by_type.csv`, and we recompute the q-values from the p-values. We assert that our units, eligible tickets, repeats, ratios, and q-values match those two tables.

In [8]:
min_eligible, alpha = summary['definitions']['min_eligible'], summary['definitions']['alpha']
fleet_ratio = within_30.sum() / expected_30  # the fleet's observed ÷ random-timing repeats
NOT_TYPES = ['No device match', 'Unknown']  # shown as rows of their own, never tested, as neither is a telemetry type


def bh(p):
    """Benjamini–Hochberg q-values, computed as in followups.py."""
    p = np.asarray(p, float)
    order = np.argsort(p, kind='mergesort')
    q = np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    out = np.empty(len(p))
    out[order] = np.minimum(q, 1.0)
    return out


def tested(key, name):
    """The units the script tests, recomputed from rp_pairs.csv, with the script's bootstrap interval and p-value."""
    script = R.read_csv(OUT, f'followups/{name}').set_index(key)
    units = (pairs.assign(repeat=within_30).groupby(key)
             .agg(eligible=('case_id', 'size'), repeats=('repeat', 'sum'), expected=('null_p30', 'sum')))
    units = units[(units.eligible >= min_eligible) & ~units.index.isin(NOT_TYPES)].copy()
    units['ratio_to_fleet'] = units.repeats / units.expected / fleet_ratio
    rows = script.loc[units.index]
    units[['lo', 'hi', 'p']] = rows[['srr_lo', 'srr_hi', 'srr_p']].to_numpy()  # from the script's bootstrap
    units['q'] = bh(units.p)
    units['stands_out'] = ((units.lo > 1) | (units.hi < 1)) & (units.q < alpha)
    agrees = (set(units.index) == set(script.index[~script.pooled & ~script.index.isin(NOT_TYPES)])
              and (units.eligible == rows.eligible).all() and (units.repeats == rows.repeats_30).all()
              and np.allclose(units.ratio_to_fleet, rows.srr) and np.allclose(units.q, rows.srr_q))
    return units, agrees


models, models_agree = tested('model', 'rp_by_model.csv')
types, types_agree = tested('telemetry_type', 'rp_by_type.csv')
n_stand_out = int(models.stands_out.sum() + types.stands_out.sum())
assert models_agree and types_agree, 'our units or values differ from rp_by_model.csv or rp_by_type.csv'
print('Our units, eligible tickets, repeats, ratios, and q-values match rp_by_model.csv and rp_by_type.csv.')
for label, units in (('Models', models), ('Telemetry types', types)):
    excl = units[(units.lo > 1) | (units.hi < 1)]
    print(f'{label} whose interval excludes 1: ' + '; '.join(  # three decimals, so a bound near 1 shows its side
        f'{u} {r.ratio_to_fleet:.2f} ({R.span(r.lo, r.hi, 3)}, q = {r.q:.2f})' for u, r in excl.iterrows()))
pd.DataFrame({label: {'tested': len(units), 'eligible tickets': int(units.eligible.sum()),
                      'ratio to the fleet, range': R.span(units.ratio_to_fleet.min(), units.ratio_to_fleet.max(), 2),
                      'interval excludes 1': int(((units.lo > 1) | (units.hi < 1)).sum()),
                      'smallest q': R.num(units.q.min(), 2), 'clearly stand out': int(units.stands_out.sum())}
              for label, units in (('models', models), ('telemetry types', types))}).T

Our units, eligible tickets, repeats, ratios, and q-values match rp_by_model.csv and rp_by_type.csv.
Models whose interval excludes 1: LC-30 0.73 (0.502–0.966, q = 0.17); LVD-Remote 1.16 (1.003–1.346, q = 0.17); LVRR-WP-2.0 0.91 (0.838–0.996, q = 0.17)
Telemetry types whose interval excludes 1: scrubex-2 1.20 (1.031–1.412, q = 0.12); scrubex2.0-3 0.90 (0.817–0.982, q = 0.12)


,tested,eligible tickets,"ratio to the fleet, range",interval excludes 1,smallest q,clearly stand out
models,12,6103,0.73–1.16,3,0.17,0
telemetry types,11,4986,0.83–1.20,2,0.12,0


## Check against the report

In [9]:
R.check([
    ('Eligible tickets', '6,665', R.num(n_eligible)),
    ("Repeat window, days (from the script's repeat_30 flags)", '30', R.num(window_30)),
    ('Share that repeat within 30 days', '23.6%', R.pct(share_30, 1)),
    ("Short repeat window, days (from the script's repeat_7 flags)", '7', R.num(window_7)),
    ('Share that repeat within 7 days', '9.5%', R.pct(share_7, 1)),
    ("30-day share vs. random timing's share", '1.53', R.num(ratio_30, 2)),
    ("7-day share vs. random timing's share", '2.30', R.num(ratio_7, 2)),
    ('30-day share without monitoring tickets', '22.0%', R.pct(share_no_monitoring, 1)),
    ('Repeats', '1,574', R.num(n_repeats)),
    ('Repeats: same Category', '31%', R.pct(class_share['Same Category'])),
    ('Repeats: same request group, another Category', '10%', R.pct(class_share['Same group, other Category'])),
    ('Repeats: different group', '32%', R.pct(class_share['Different group'])),
    ('Repeats: monitoring, No Trouble Found, or a blank Category involved', '28%', R.pct(share_process)),
    ('Shortest gap bin over all pairs', 'a day or less', f'{in_words(nearest_days)} or less'),
    ('All pairs a day or less apart that share a Category', '46%', R.pct(share_nearest)),
    ('Longest gap bin over all pairs', 'over six months', f'over {in_words(farthest_days)}'),
    ('All pairs over six months apart that share a Category', '27%', R.pct(share_farthest)),
    ('Eligible tickets a model or telemetry type needs to be tested', '100', R.num(min_eligible)),
    ('Models or telemetry types with 100+ eligible tickets that clearly stand out', 'No',
     'No' if n_stand_out == 0 else R.num(n_stand_out)),
    ('Models tested (Definitions, Clearly stands out)', '12', R.num(len(models))),
    ('Telemetry types tested (Definitions, Clearly stands out)', '11', R.num(len(types))),
]);

,Finding,Report,Recomputed,Match
0,Eligible tickets,"6,665","6,665",yes
1,"Repeat window, days (from the script's repeat_30 flags)",30,30,yes
2,Share that repeat within 30 days,23.6%,23.6%,yes
3,"Short repeat window, days (from the script's repeat_7 flags)",7,7,yes
4,Share that repeat within 7 days,9.5%,9.5%,yes
5,30-day share vs. random timing's share,1.53,1.53,yes
6,7-day share vs. random timing's share,2.30,2.30,yes
7,30-day share without monitoring tickets,22.0%,22.0%,yes
8,Repeats,"1,574","1,574",yes
9,Repeats: same Category,31%,31%,yes


All 21 values match the report.
